# Dependence measures

For selected numeric column pairs, the dependence screen calculates
Pearson, Spearman, and Chatterjee's ξ in both directions. It reports
statistical associations; these measures alone do not show that one column
determines another.

## Three coefficients

- **Pearson** — measures linear association.
- **Spearman** — measures monotone association using average ranks for ties.
- **Chatterjee's ξ** (JASA 2021) — an asymmetric measure of dependence
  that can detect non-monotone patterns. The screen calculates both
  directions because `ξ(x → y)` and `ξ(y → x)` can differ.

When x has ties, the estimator randomizes their order using a fixed seed.
Sorting ties by row order can introduce dependence on row position; for
example, a block ID compared with a row counter can produce a high ξ on
independent values. ξ is undefined if either column is constant or if
fewer than 30 pairs are available.

## Screening and caps

The pass scores all numeric pairs using normalized mutual information over
encoded bins from up to 2,048 rows. It then calculates Pearson, Spearman,
and ξ on up to 496 pairs (`standard`) or 2,016 (`deep`), using at most
20,000 or 50,000 sampled rows.
Each tested pair also gets a p-value, combined from its component
tests, and a Benjamini–Hochberg q-value computed over the pairs the
screen tested in the run. A pair is stated as a finding when q ≤ 0.05:
in the strong tier when its strongest signal reaches 0.5, and in a
weak tier when it is significant but below that bar. Weak-tier
findings keep their full evidence in the JSON report; the digest
collapses them to one line giving the count and the strongest
examples. The top 20 stated pairs are kept, and pairs with signal
≥ 0.98 become redundancy links that feed the connected-groups view.
Every computed pair, stated or not, appears in the report's
`associations` table with its coefficients, pair-complete count,
p-value, q-value, and tier.


In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(31)
n = 4_000
x = rng.uniform(-1, 1, n)
df = pl.DataFrame({
    "x": x,
    "linear": 3 * x + 1,            # Pearson ~ 1
    "monotone_bend": x ** 3,        # Spearman 1, Pearson < 1
    "parabola": x ** 2,             # xi high, Pearson ~ 0
})

report = onceover.profile(df)
for f in report.findings:
    if f["type"] == "statistical":
        m = f["evidence"][0]["metrics"]
        a, b = m["variables"]
        print(f"{a:>14s} × {b:<14s} pearson={m['pearson']:+.3f} "
              f"spearman={m['spearman']:+.3f} xi(a→b)={m['xi_ab']:.3f} xi(b→a)={m['xi_ba']:.3f}")

In the controlled examples, Pearson is highest for a linear pattern.
Spearman remains high for a monotone curve. For the non-monotone
relationship `y = x²`, the two ξ directions differ: x determines y, but y
does not determine x.

## Interpretation

Association coefficients describe patterns in the evaluated pairs. They do
not establish a functional mapping or a causal effect. A low ξ on a sample
means no strong ξ signal was detected at that sample size; coverage records
the scope
([](../usage/coverage.ipynb)).

## Related questions

A coefficient answers a pairwise association question. Other methods ask
how that relationship behaves or how it enters a target model:

- [Target shapes](target_shapes.ipynb): does a line, curve, or step describe
  one predictor's association with a designated response?
- [Predictor interactions](interactions.ipynb): does a pair act jointly
  beyond additive effects under a fitted target model?
- [Period associations](period_associations.ipynb): does the pair's
  correlation change, even reverse sign, between periods?
- [Functional dependencies](functional_dependencies.ipynb): do equal
  inputs map to equal outputs, or does a formula hold within tolerance?

These statements have different evidence. A large coefficient does not
upgrade a prediction to an exact constraint, and opposite ξ directions
need not be equal.
